# SAMATRIX RESUMEFORGE 2026: END-TO-END NLP RESUME CLASSIFICATION
### A Complete Phase-Wise NLP Lifecycle (Total: 70 Marks)
**Workflow: Problem Definition → Data Gathering → Quality Audit → EDA → Text Preprocessing → Stratified Split → Feature Engineering → Classical & Deep Learning Models → Multi-Metric Evaluation → Error Analysis → Inference Pipeline & Testing**

This notebook fulfills all 12 stages specified in the official **Problem statement and Guidelines for Hackathon.docx** with zero data leakage, reproducible preprocessing, and comprehensive diagnostics.

## 📌 70-Mark Evaluation Rubric & Stage Mapping
| Step | Stage | What You Should Do | Marks |
| :---: | :--- | :--- | :---: |
| 1 | Problem Understanding | Define input, output, classes, constraints and success metric | 5 |
| 2 | Data Gathering | Load dataset; inspect files, columns, labels and raw resume text | 5 |
| 3 | Data Quality Check | Missing values, duplicates, corrupted/empty resumes, label inconsistencies, leakage | 5 |
| 4 | EDA | Class distribution, text length, vocabulary, WordCloud, unigram/bigram/trigram analysis | 10 |
| 5 | Text Preprocessing | Normalize text carefully; remove noise while preserving meaningful resume info | 10 |
| 6 | Train/Val/Test Split | Split before fitting vocabulary/vectorizers to prevent data leakage | 5 |
| 7 | Feature Engineering | TF-IDF baseline; optionally embeddings; derive useful text statistics | 5 |
| 8 | ML Models | Logistic Regression / Linear SVM / Naive Bayes or another justified baseline | 5 |
| 9 | DL Model | Embedding + Neural/Dense/LSTM/GRU model or justified neural approach | 5 |
| 10 | Evaluation | Accuracy + Precision/Recall/Macro & Weighted F1, confusion matrix, per-class results | 5 |
| 11 | Error Analysis | Inspect wrong predictions and identify data/model weaknesses | 5 |
| 12 | Final Pipeline & Demo | Raw resume → preprocessing → feature extraction → model → predicted category | 5 |
| **Total** | | **All 12 Stages Covered** | **70 Marks** |

In [ ]:
# Setup Environment and Imports
import os
import re
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from pathlib import Path

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuration Constants
RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
DATA_CSV_PATH = "Dataset/Resume/Resume.csv"

# Create artifact directories
os.makedirs("artifacts/models", exist_ok=True)
os.makedirs("artifacts/figures", exist_ok=True)
os.makedirs("artifacts/reports", exist_ok=True)

print("Environment configured successfully!")

---
## 🎯 STAGE 1: Problem Understanding (5 Marks)
- **Problem Statement**: Given raw unstructured resume text, predict its industry domain category among 24 classes.
- **Input Data**: Raw resume strings extracted from CV documents (`Resume_str`).
- **Target Variable**: `Category` (Categorical string representing one of 24 industry sectors).
- **Core Constraints**: 
  - Zero data leakage: Tokenizers, IDF frequencies, and vectorizers must be fitted **strictly** on the training split.
  - Domain preservation: Technical tokens (`C++`, `Python`, `AWS`, `SQL`, `.NET`, `CI/CD`) must be preserved during cleaning.
- **Primary Optimization Metric**: **Macro-F1 Score** (treats all 24 classes equally regardless of class imbalance), alongside Weighted-F1 and Accuracy.

---
## 📥 STAGE 2: Data Gathering & Ingestion (5 Marks)
Load the raw dataset from `Dataset/Resume/Resume.csv`, inspect columns, data types, and initial records.

In [ ]:
# Stage 2: Load raw dataset
df_raw = pd.read_csv(DATA_CSV_PATH)
print(f"Total Records Ingested : {len(df_raw)}")
print(f"Dataset Columns        : {df_raw.columns.tolist()}")
print(f"Target Categories Count: {df_raw['Category'].nunique()} unique classes\n")
print("First 3 records preview:")
df_raw.head(3)

---
## 🔍 STAGE 3: Data Quality Check & Leakage Prevention (5 Marks)
Examine missing values, duplicate IDs, exact duplicate resume strings, label consistency, and text length distributions.

In [ ]:
# Stage 3: Data Quality Audit
missing_counts = df_raw.isnull().sum()
duplicate_ids = df_raw['ID'].duplicated().sum()
duplicate_resumes = df_raw['Resume_str'].duplicated().sum()

char_lengths = df_raw['Resume_str'].astype(str).apply(len)
word_lengths = df_raw['Resume_str'].astype(str).apply(lambda s: len(s.split()))
empty_or_tiny = (word_lengths < 20).sum()

print("==================================================")
print("DATA QUALITY AUDIT REPORT")
print("==================================================")
print(f"• Total Records               : {len(df_raw)}")
print(f"• Missing Values              :\n{missing_counts}")
print(f"• Duplicate IDs               : {duplicate_ids}")
print(f"• Duplicate Resume Texts      : {duplicate_resumes}")
print(f"• Near-Empty Resumes (<20 w)  : {empty_or_tiny}")
print(f"• Character Length Statistics : Min={char_lengths.min()}, Mean={char_lengths.mean():.1f}, Max={char_lengths.max()}")
print(f"• Word Count Statistics       : Min={word_lengths.min()}, Mean={word_lengths.mean():.1f}, Max={word_lengths.max()}")
print("==================================================")

# Normalize string columns
df = df_raw.copy()
df['Resume_str'] = df['Resume_str'].astype(str)
df['Category'] = df['Category'].astype(str).str.strip().str.upper()

---
## 📊 STAGE 4: Comprehensive Exploratory Data Analysis (EDA) (10 Marks)
Investigate:
1. **Class Distribution**: Category representation across all 24 classes.
2. **Text Length Distributions**: Raw characters and word length histograms with KDE.
3. **Corpus WordCloud**: High-frequency terms across resumes.
4. **N-Gram Analysis**: Top unigrams, bigrams, and trigrams.

In [ ]:
# 4.1 Class Distribution
plt.figure(figsize=(14, 6))
cat_order = df['Category'].value_counts()
ax = sns.barplot(x=cat_order.index, y=cat_order.values, palette="mako")
plt.title("Stage 4: Resume Distribution Across 24 Industry Categories", fontsize=14, pad=12)
plt.xlabel("Category", fontsize=11)
plt.ylabel("Number of Resumes", fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=8, color='black',
                xytext=(0, 4), textcoords='offset points')
plt.tight_layout()
plt.savefig("artifacts/figures/class_distribution.png", dpi=300)
plt.show()

In [ ]:
# 4.2 Text Length Distributions (Characters and Words)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(char_lengths, bins=40, kde=True, ax=axes[0], color="teal")
axes[0].set_title("Distribution of Resume Character Lengths", fontsize=12)
axes[0].set_xlabel("Number of Characters")
axes[0].set_ylabel("Frequency")

sns.histplot(word_lengths, bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of Resume Word Counts", fontsize=12)
axes[1].set_xlabel("Number of Words")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("artifacts/figures/text_length_distribution.png", dpi=300)
plt.show()

In [ ]:
# 4.3 Corpus WordCloud
all_corpus_text = " ".join(df['Resume_str'])
wc = WordCloud(
    width=1200,
    height=500,
    background_color="white",
    colormap="viridis",
    max_words=120,
    random_state=RANDOM_STATE
).generate(all_corpus_text)

plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Corpus-Wide Key Terms WordCloud", fontsize=14, pad=12)
plt.tight_layout()
plt.savefig("artifacts/figures/wordcloud_overall.png", dpi=300)
plt.show()

In [ ]:
# 4.4 N-Grams Analysis: Top Unigrams, Bigrams, and Trigrams
def extract_ngrams(text_series, n, top_k=15):
    all_tokens = " ".join(text_series).lower()
    # Strip basic noise
    words = re.findall(r'\b[a-z]{2,}\b', all_tokens)
    stopwords = set(["and", "the", "to", "of", "in", "for", "with", "a", "an", "on", "as", "at", "by", "from", "is", "or", "that", "this"])
    words = [w for w in words if w not in stopwords]
    
    if n == 1:
        ngrams = words
    else:
        ngrams = [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]
    return Counter(ngrams).most_common(top_k)

top_u = extract_ngrams(df['Resume_str'], 1, 15)
top_b = extract_ngrams(df['Resume_str'], 2, 15)
top_t = extract_ngrams(df['Resume_str'], 3, 15)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

u_lbl, u_val = zip(*top_u)
sns.barplot(x=list(u_val), y=list(u_lbl), ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 15 Unigrams", fontsize=12)
axes[0].set_xlabel("Frequency")

b_lbl, b_val = zip(*top_b)
sns.barplot(x=list(b_val), y=list(b_lbl), ax=axes[1], palette="Greens_r")
axes[1].set_title("Top 15 Bigrams", fontsize=12)
axes[1].set_xlabel("Frequency")

t_lbl, t_val = zip(*top_t)
sns.barplot(x=list(t_val), y=list(t_lbl), ax=axes[2], palette="Oranges_r")
axes[2].set_title("Top 15 Trigrams", fontsize=12)
axes[2].set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("artifacts/figures/ngrams_analysis.png", dpi=300)
plt.show()

---
## 🧹 STAGE 5: Text Preprocessing & Technical Token Preservation (10 Marks)
A crucial guideline requirement:
> *"Apply preprocessing deliberately. Do not remove information blindly; resume text contains useful skills, technologies, certifications, education terms, job titles, and experience indicators. Preserve meaningful technical tokens such as Python, C++, SQL, AWS, TensorFlow, Java, .NET, NLP, etc."*

Our custom cleaning pipeline protects multi-character tokens like `C++`, `C#`, `.NET`, `Node.js`, and `CI/CD` before cleaning noise.

In [ ]:
# Stage 5: Reproducible Text Preprocessor
TOKEN_MAP = {
    r"\bc\+\+\b": "cplusplus",
    r"\bc#\b": "csharp",
    r"\b\.net\b": "dotnet",
    r"\bnode\.js\b": "nodejs",
    r"\bvue\.js\b": "vuejs",
    r"\breact\.js\b": "reactjs",
    r"\bci\/cd\b": "cicd",
    r"\bpower\s+bi\b": "powerbi",
    r"\bms\s+excel\b": "msexcel",
    r"\bms\s+office\b": "msoffice",
    r"\bspring\s+boot\b": "springboot",
    r"\bmachine\s+learning\b": "machinelearning",
    r"\bdeep\s+learning\b": "deeplearning",
    r"\bdata\s+science\b": "datascience",
    r"\bcloud\s+computing\b": "cloudcomputing"
}

STOPWORDS = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
    "any", "are", "as", "at", "be", "because", "been", "before", "being", "below",
    "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down",
    "during", "each", "few", "for", "from", "further", "had", "has", "have",
    "having", "he", "her", "here", "hers", "herself", "him", "himself", "his",
    "how", "i", "if", "in", "into", "is", "it", "its", "itself", "me", "more",
    "most", "my", "myself", "no", "nor", "not", "of", "off", "on", "once", "only",
    "or", "other", "ought", "our", "ours", "ourselves", "out", "over", "own",
    "same", "she", "should", "so", "some", "such", "than", "that", "the", "their",
    "theirs", "them", "themselves", "then", "there", "these", "they", "this",
    "those", "through", "to", "too", "under", "until", "up", "very", "was", "we",
    "were", "what", "when", "where", "which", "while", "who", "whom", "why", "with",
    "would", "you", "your", "yours", "yourself", "yourselves"
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 1. Lowercase
    t = text.lower()
    # 2. Protect technical terms
    for pat, rep in TOKEN_MAP.items():
        t = re.sub(pat, rep, t)
    # 3. Strip HTML markup
    t = re.sub(r"<[^>]+>", " ", t)
    # 4. Remove URLs, emails, phone numbers
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", " ", t)
    t = re.sub(r"\+?\d[\d\s\-\(\)]{7,}\d", " ", t)
    # 5. Clean punctuation & numbers
    t = re.sub(r"[^a-zA-Z0-9\s]", " ", t)
    # 6. Filter stopwords
    words = [w for w in t.split() if w not in STOPWORDS and len(w) >= 2 and not w.isdigit()]
    return " ".join(words)

# Apply preprocessing to corpus
print("Cleaning all 2,484 resumes with reproducible preprocessor...")
df['Clean_text'] = df['Resume_str'].apply(clean_text)
print("Preprocessing completed!")
print(f"Sample Raw Snippet   : {df['Resume_str'].iloc[0][:120]}...")
print(f"Sample Clean Snippet : {df['Clean_text'].iloc[0][:120]}...")

---
## 🛡️ STAGE 6: Leakage-Free Stratified Train / Validation / Test Split (5 Marks)
Split dataset into:
- **70% Training** (1,738 resumes)
- **15% Validation** (373 resumes)
- **15% Test** (373 resumes)
Strictly partitioned **before** fitting vectorizers to guarantee zero test leakage.

In [ ]:
# Stage 6: Stratified Splitting
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])
joblib.dump(le, "artifacts/models/label_encoder.joblib")

# Split Train (70%) and Temp (30%)
df_train, df_temp = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['Category_Encoded'],
    random_state=RANDOM_STATE
)

# Split Temp into Val (15%) and Test (15%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    stratify=df_temp['Category_Encoded'],
    random_state=RANDOM_STATE
)

print(f"• Training Set   : {len(df_train):4d} resumes ({len(df_train)/len(df)*100:.1f}%)")
print(f"• Validation Set : {len(df_val):4d} resumes ({len(df_val)/len(df)*100:.1f}%)")
print(f"• Test Set       : {len(df_test):4d} resumes ({len(df_test)/len(df)*100:.1f}%)")

---
## ⚙️ STAGE 7: Feature Engineering (TF-IDF N-Grams) (5 Marks)
Fit `TfidfVectorizer` **only** on the training corpus:
- `ngram_range=(1, 2)`: Captures single domain words and compound phrases (e.g. *machine learning*, *tax audit*).
- `sublinear_tf=True`: Replaces $tf$ with $1 + \log(tf)$ to dampen redundant token repetitions.
- `max_features=12000`, `min_df=2`, `max_df=0.95`.

In [ ]:
# Stage 7: Feature Engineering
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=12000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode'
)

# Fit strictly on Training data
X_train = vectorizer.fit_transform(df_train['Clean_text'])
X_val = vectorizer.transform(df_val['Clean_text'])
X_test = vectorizer.transform(df_test['Clean_text'])

y_train = df_train['Category_Encoded'].values
y_val = df_val['Category_Encoded'].values
y_test = df_test['Category_Encoded'].values

joblib.dump(vectorizer, "artifacts/models/tfidf_vectorizer.joblib")
print(f"Fitted TF-IDF Vocabulary Size: {len(vectorizer.vocabulary_)} features")
print(f"Training Feature Matrix Shape: {X_train.shape}")
print(f"Validation Feature Matrix    : {X_val.shape}")
print(f"Test Feature Matrix          : {X_test.shape}")

---
## 🤖 STAGE 8 & 9: Classical ML & Deep Neural Network Models (10 Marks)
We benchmark four diverse architectures:
1. **Multinomial Naive Bayes** (Fast classical text baseline)
2. **Logistic Regression** (Balanced multi-class linear classifier)
3. **Linear Support Vector Classifier** (Calibrated Linear SVM)
4. **Deep Neural Network (MLP)** (Deep Dense network: $256 \rightarrow 128$ hidden units, ReLU, Adam, early stopping)

In [ ]:
# Stage 8 & 9: Model Training and Validation Comparison
candidate_models = {
    "Multinomial_Naive_Bayes": MultinomialNB(alpha=0.1),
    "Logistic_Regression": LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Linear_SVM": CalibratedClassifierCV(
        LinearSVC(C=0.5, class_weight="balanced", random_state=RANDOM_STATE, max_iter=2000),
        cv=3
    ),
    "Deep_Neural_Network_MLP": MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        batch_size=64,
        learning_rate="adaptive",
        max_iter=200,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

val_results = {}
best_model_name = None
best_macro_f1 = -1.0
best_model = None

for name, model in candidate_models.items():
    print(f"Training {name} on {X_train.shape[0]} samples...")
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred_val)
    macro_f1 = f1_score(y_val, y_pred_val, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_val, y_pred_val, average="weighted", zero_division=0)
    
    val_results[name] = {
        "Val Accuracy": f"{acc*100:.2f}%",
        "Val Macro-F1": round(macro_f1, 4),
        "Val Weighted-F1": round(weighted_f1, 4)
    }
    
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_model_name = name
        best_model = model

# Display Comparison
df_comparison = pd.DataFrame(val_results).transpose()
print("\n" + "="*60)
print(f"BEST MODEL SELECTED: {best_model_name} (Validation Macro-F1: {best_macro_f1:.4f})")
print("="*60)
display(df_comparison)

# Save best model
joblib.dump(best_model, "artifacts/models/best_model.joblib")
with open("artifacts/reports/model_comparison.json", "w") as f:
    json.dump({"comparison": val_results, "best_model": best_model_name}, f, indent=4)
print("Saved best model to artifacts/models/best_model.joblib")

---
## 🎯 STAGE 10: Multi-Metric Evaluation on Untouched Test Set (5 Marks)
Evaluate the winning model on the **373 test resumes** that were held out.
Compute Accuracy, Precision, Recall, Macro-F1, Weighted-F1, and plot the normalized Confusion Matrix.

In [ ]:
# Stage 10: Final Test Set Evaluation
target_names = list(le.classes_)
y_test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_p = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_r = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)

print("="*55)
print("STAGE 10: UNTOUCHED TEST SET EVALUATION")
print("="*55)
print(f"• Test Accuracy        : {test_acc*100:.2f}%")
print(f"• Test Macro-Precision : {test_macro_p:.4f}")
print(f"• Test Macro-Recall    : {test_macro_r:.4f}")
print(f"• Test Macro-F1        : {test_macro_f1:.4f}")
print(f"• Test Weighted-F1     : {test_weighted_f1:.4f}")
print("="*55)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_test_pred)
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(15, 13))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
    cbar_kws={'label': 'Proportion of Actual Class'}
)
plt.title(f"Normalized Confusion Matrix — {best_model_name} (Test Set)", fontsize=14, pad=12)
plt.xlabel("Predicted Category", fontsize=11)
plt.ylabel("Actual Category", fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("artifacts/figures/test_confusion_matrix.png", dpi=300)
plt.show()

# Save per-class classification report
cls_report = classification_report(y_test, y_test_pred, target_names=target_names, output_dict=True)
pd.DataFrame(cls_report).transpose().to_csv("artifacts/reports/classification_report.csv")
print("Saved per-class classification report to artifacts/reports/classification_report.csv")

---
## 🔍 STAGE 11: In-Depth Error Analysis (5 Marks)
Examine all test set misclassifications:
- Compare Actual vs. Predicted labels
- Inspect model confidence scores
- Identify frequent cross-category confusion pairs
- Diagnose failure causes (e.g. `CONSULTANT` ↔ `INFORMATION-TECHNOLOGY`, `FINANCE` ↔ `ACCOUNTANT`)

In [ ]:
# Stage 11: Diagnostic Error Analysis
test_raw = df_test['Resume_str'].tolist()
test_clean = df_test['Clean_text'].tolist()
test_actual = df_test['Category'].tolist()
test_pred_labels = [target_names[i] for i in y_test_pred]

if hasattr(best_model, "predict_proba"):
    test_probs = best_model.predict_proba(X_test)
else:
    test_probs = np.zeros((len(y_test), len(target_names)))

errors = []
for i in range(len(y_test)):
    if test_actual[i] != test_pred_labels[i]:
        conf = float(np.max(test_probs[i]))
        errors.append({
            "Actual": test_actual[i],
            "Predicted": test_pred_labels[i],
            "Confidence": round(conf, 4),
            "Word_Count": len(test_clean[i].split()),
            "Snippet": test_clean[i][:180]
        })

df_err = pd.DataFrame(errors)
print(f"Total Misclassifications Diagnosed: {len(df_err)} out of {len(y_test)} test samples\n")

print("Top 8 Most Frequent Confused Pairs:")
confusion_pairs = (
    df_err.groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
    .sort_values(by="Count", ascending=False)
)
display(confusion_pairs.head(8))

df_err.to_csv("artifacts/reports/error_analysis.csv", index=False)
print("Saved detailed error analysis to artifacts/reports/error_analysis.csv")

---
## 🚀 STAGE 12: End-to-End Prediction Pipeline & Real-World Testing (5 Marks)
Implement a self-contained inference pipeline that accepts raw uncleaned resume text, applies identical transformations, and outputs:
- Top Predicted Category
- Model Confidence Score
- Top 3 Candidate Classes with Probabilities
- Key Matched Domain Keywords

In [ ]:
# Stage 12: Final Inference Pipeline
class ResumePipeline:
    def __init__(self, model, vec, encoder):
        self.model = model
        self.vec = vec
        self.encoder = encoder
        self.classes = list(encoder.classes_)
        self.feature_names = np.array(vec.get_feature_names_out())
        
    def predict(self, raw_text: str, top_k=3):
        # 1. Reproducible preprocessing
        cleaned = clean_text(raw_text)
        # 2. Vectorization
        feat = self.vec.transform([cleaned])
        # 3. Probabilities
        if hasattr(self.model, "predict_proba"):
            probs = self.model.predict_proba(feat)[0]
        else:
            probs = np.zeros(len(self.classes))
            probs[self.model.predict(feat)[0]] = 1.0
            
        top_idx = np.argsort(probs)[::-1][:top_k]
        top_candidates = [
            (self.classes[i], f"{probs[i]*100:.1f}%")
            for i in top_idx
        ]
        
        # Matched domain keywords
        csr = feat.tocsr()
        kw = [self.feature_names[csr.indices[j]] for j in np.argsort(csr.data)[::-1][:6]] if len(csr.data) > 0 else []
        
        return {
            "predicted_category": self.classes[top_idx[0]],
            "confidence": f"{probs[top_idx[0]]*100:.1f}%",
            "top_candidates": top_candidates,
            "matched_keywords": kw
        }

# Instantiate inference engine
pipeline = ResumePipeline(best_model, vectorizer, le)

# Test with 4 unseen real-world resumes
sample_resumes = [
    {
        "role": "Cloud Architect & Full-Stack Developer",
        "text": """Senior Software Engineer with 7+ years in cloud microservices.
        Expert in Python, FastAPI, React.js, TypeScript, PostgreSQL, and Docker containerization.
        Architected AWS infrastructure (EC2, S3, RDS, Lambda) with automated CI/CD pipelines."""
    },
    {
        "role": "Certified Public Accountant",
        "text": """Senior Accountant & CPA with 8 years of internal audit experience.
        Specialized in general ledger reconciliations, balance sheet preparation, tax filings,
        and GAAP compliance. Advanced QuickBooks, SAP ERP, and financial modeling in Excel."""
    },
    {
        "role": "Human Resources & Talent Acquisition Specialist",
        "text": """HR Manager with 6 years leading global talent acquisition and employee relations.
        Proficient in Workday HRIS, BambooHR, LinkedIn Recruiter, onboarding, and payroll compliance."""
    },
    {
        "role": "Executive Sous Chef",
        "text": """Executive Chef with 10 years in Michelin-guide fine dining culinary arts.
        Specialized in menu creation, farm-to-table sourcing, pastry, and ServSafe HACCP safety standards."""
    }
]

print("==========================================================================")
print("TESTING PIPELINE ON UNSEEN REAL-WORLD RESUMES")
print("==========================================================================")
for idx, s in enumerate(sample_resumes, 1):
    res = pipeline.predict(s['text'])
    print(f"\n[Test Case {idx}] Input Role: {s['role']}")
    print(f" -> Predicted Category : {res['predicted_category']} (Confidence: {res['confidence']})")
    print(f" -> Top 3 Candidates   : {res['top_candidates']}")
    print(f" -> Key Terms Detected : {res['matched_keywords']}")
print("\n==========================================================================")
print("All 12 Stages Successfully Completed!")